In [48]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urlparse


def get_domain_age_and_registration(url):

    try:
        # Extraer el dominio base de la URL
        parsed_url = urlparse(url)
        domain = parsed_url.netloc
        if domain.startswith("www."):
            domain = domain[4:]

        # URL de consulta en Whois.com
        whois_url = f"https://www.whois.com/whois/{domain}"

        # Realizar la solicitud HTTP
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
        }
        response = requests.get(whois_url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"Error HTTP al acceder a Whois.com para {domain}: {response.status_code}")
            return -1, 0

        # Parsear el contenido HTML
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Buscar la información WHOIS en el elemento <pre>
        pre_element = soup.find("pre", id="registryData")
        if not pre_element:
            print(f"No se encontró el elemento <pre> con id='registryData' para {domain}")
            return -1, 0

        # Extraer texto y buscar "Creation date"
        pre_text = pre_element.get_text()
        
        # Determinar si el dominio está registrado
        is_registered = 1 if "Creation date" in pre_text else 0
        
        # Si está registrado, extraer la fecha de creación
        creation_date_line = next((line for line in pre_text.splitlines() if "Creation date" in line), None)
        if creation_date_line:
            creation_date_str = creation_date_line.split("Creation date:")[1].strip().split(" ")[0]  # Tomar solo la fecha
            creation_date = datetime.strptime(creation_date_str, "%Y-%m-%d")

            # Calcular la edad del dominio
            age_days = (datetime.now() - creation_date).days
        else:
            age_days = -1  # No se encontró la fecha de creación

        return age_days, is_registered

    except Exception as e:
        print(f"Error al obtener la información del dominio para '{url}': {e}")
        return -1, 0

# Ejemplo de uso
url = "https://subaru.cl/"
result = get_domain_age_and_registration(url)
print(f"Resultado: {result[0]}")


Resultado: 9913


In [ ]:
import metodos_caracteristicas as mt


NameError: name 'url' is not defined

In [9]:
mt.nb_hyperlinks(url)

240

In [10]:
mt.domain_in_title(url)

1

In [11]:
mt.get_domain_age(url)

9913

In [12]:
print(mt.domain_with_copyright(url))

0


In [13]:
print(mt.whois_registered_domain(url))

1


In [14]:
print(mt.dns_record(url))

1


In [15]:
print(0.3 + 1.7 + 1.3 + 0.2 + 1.0 + 0.1 + 0.2)

4.8


In [ ]:
################################################################################################################################################################################################
################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################

In [65]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import tldextract
import wordninja

def analyze_page_content(url):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        # Realizar la solicitud GET con el User-Agent
        response = requests.get(url, headers=headers, timeout=10)
        
        # Verificar el código de respuesta HTTP
        if response.status_code != 200:
            return {
                "nb_hyperlinks": 0,
                "domain_in_title": 0,
                "domain_with_copyright": 0
            }

        # Parsear el contenido HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # --- Calcular nb_hyperlinks ---
        href_count = len(soup.find_all('a', href=True))  # Contar enlaces <a> con href
        src_count = len(soup.find_all(src=True))  # Contar etiquetas con src (ej., <img>, <script>, etc.)
        nb_hyperlinks = href_count + src_count

        # --- Verificar domain_in_title ---
        # Extraer el dominio base
        extracted = tldextract.extract(url)
        base_domain = extracted.domain  # Nombre base del dominio (sin TLD)
        palabras_base_domain = wordninja.split(base_domain)  # Dividir en palabras

        # Extraer el título de la página
        title = soup.title.string if soup.title else ""
        domain_in_title = 0
        for palabra in palabras_base_domain:
            if palabra.lower() in title.lower():
                domain_in_title = 1
                break

        # --- Verificar domain_with_copyright ---
        # Extraer el dominio base completo (host)
        domain_match = re.search(r'://(www\.)?([^/]+)', url)
        domain_full = domain_match.group(2) if domain_match else ""

        # Buscar símbolos de copyright en el contenido
        match = re.search(r'(\N{COPYRIGHT SIGN}|\N{TRADE MARK SIGN}|\N{REGISTERED SIGN}|&copy;|&trade;|&reg;)', response.text)
        domain_with_copyright = 0
        if match:
            # Extraer contexto cercano al símbolo
            context_range = 300
            _copyright = response.text[max(0, match.span()[0]-context_range):match.span()[0]+context_range]
            if domain_full.lower() in _copyright.lower():
                domain_with_copyright = 1

        # Retornar resultados como diccionario
        return {
            "nb_hyperlinks": nb_hyperlinks,
            "domain_in_title": domain_in_title,
            "domain_with_copyright": domain_with_copyright
        }

    except requests.exceptions.RequestException as e:
        print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }
    except Exception as e:
        print(f"Error general procesando la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

# Ejemplo de uso
result = analyze_page_content("https://escuelamartinpescador.cl/")
print(result)


{'nb_hyperlinks': 73, 'domain_in_title': 1, 'domain_with_copyright': 1}


In [25]:
from datetime import datetime
from urllib.parse import urlparse
from publicsuffix2 import get_sld
import whois
import re

def analyze_domain_whois(url):
    """
    Combina las funciones para calcular la edad del dominio y verificar si está registrado mediante WHOIS.

    Args:
        url (str): URL del dominio a analizar.

    Returns:
        dict: Diccionario con:
            - "domain_age" (int): Edad del dominio en días (-1 si ocurre un error o no hay información).
            - "is_registered" (int): 1 si el dominio está registrado, 0 en caso contrario.
    """
    try:
        # Parsear la URL para obtener el dominio
        parsed_url = urlparse(url)
        domain = parsed_url.netloc

        # Eliminar prefijo 'www.' si existe
        if domain.startswith("www."):
            domain = domain[4:]

        # Caso especial: dominios que terminan en 'gob.cl'
        if domain.endswith("gob.cl"):
            domain = "gob.cl"

        # Obtener el dominio base
        domain = get_sld(domain)

        # Realizar la consulta WHOIS
        whois_info = whois.whois(domain)

        # Verificar si el dominio está registrado
        hostname = whois_info.domain_name
        is_registered = 0
        if isinstance(hostname, list):  # Si WHOIS devuelve una lista de dominios
            for host in hostname:
                if re.search(host.lower(), domain):
                    is_registered = 1
                    break
        elif hostname and re.search(hostname.lower(), domain):  # Si WHOIS devuelve un único dominio
            is_registered = 1

        # Calcular la edad del dominio
        creation_date = whois_info.creation_date
        domain_age = -1

        if creation_date:
            if isinstance(creation_date, list):  # Si es una lista, tomar el primer elemento
                creation_date = creation_date[0]

            if isinstance(creation_date, str):  # Convertir cadena a datetime
                creation_date = datetime.strptime(creation_date.split(" ")[0], "%Y-%m-%d")

            if isinstance(creation_date, datetime):  # Calcular la edad
                domain_age = (datetime.now() - creation_date).days

        # Retornar los resultados
        return {
            "domain_age": domain_age,
            "is_registered": is_registered
        }

    except Exception as e:
        print(f"Error al analizar el dominio '{url}': {e}")
        return {
            "domain_age": -1,
            "is_registered": 0
        }

# Ejemplo de uso
result = analyze_domain_whois(url)
print(result)


{'domain_age': 9913, 'is_registered': 1}


In [63]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import tldextract
import wordninja

def analyze_page_content(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        # Realiza la solicitud HTTP
        response = requests.get(url, headers=headers, timeout=5)
        
        if response.status_code != 200:
            return {
                "nb_hyperlinks": 0,
                "domain_in_title": 0,
                "domain_with_copyright": 0
            }

        # Parsear el HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # Contar enlaces <a> con href y etiquetas con src
        nb_hyperlinks = len(soup.find_all('a', href=True)) + len(soup.find_all(src=True))

        # Extraer el dominio base del URL
        extracted = tldextract.extract(url)
        base_domain = extracted.domain
        palabras_base_domain = set(wordninja.split(base_domain))  # Set para búsqueda eficiente

        # Obtener el título y verificar si contiene palabras del dominio
        title = soup.title.string if soup.title else ""
        domain_in_title = int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))

        # Verificar copyright
        domain_match = re.search(r'://(www\.)?([^/]+)', url)
        domain_full = domain_match.group(2) if domain_match else ""

        # Patrón para símbolos de copyright
        copyright_pattern = r'(©|\(c\)|&copy;|®|&reg;|™|&trade;)'
        match = re.search(copyright_pattern, response.text, re.IGNORECASE)

        domain_with_copyright = 0
        if match:
            # Extraer contexto cercano al símbolo
            context_range = 300
            start = max(0, match.start() - context_range)
            end = match.end() + context_range
            _copyright = response.text[start:end].lower()
            if domain_full.lower() in _copyright:
                domain_with_copyright = 1

        return {
            "nb_hyperlinks": nb_hyperlinks,
            "domain_in_title": domain_in_title,
            "domain_with_copyright": domain_with_copyright
        }

    except requests.exceptions.RequestException as e:
        print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }
    except Exception as e:
        print(f"Error general procesando la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

# Ejemplo de uso
result = analyze_page_content("https://escuelamartinpescador.cl/")
print(result)


{'nb_hyperlinks': 73, 'domain_in_title': 1, 'domain_with_copyright': 1}


In [61]:
import requests
from bs4 import BeautifulSoup
import re
from concurrent.futures import ThreadPoolExecutor
import tldextract
import wordninja

def analyze_page_content(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        # Realizar la solicitud GET con el User-Agent
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return {
                "nb_hyperlinks": 0,
                "domain_in_title": 0,
                "domain_with_copyright": 0
            }

        # Parsear el contenido HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        def calculate_hyperlinks():
            """Calcular la cantidad de enlaces y elementos con src."""
            href_count = len(soup.find_all('a', href=True))  # Contar enlaces <a> con href
            src_count = len(soup.find_all(src=True))  # Contar etiquetas con src (ej., <img>, <script>, etc.)
            return href_count + src_count

        def check_domain_in_title():
            """Verificar si el dominio está en el título."""
            # Extraer el dominio base
            extracted = tldextract.extract(url)
            base_domain = extracted.domain  # Nombre base del dominio (sin TLD)
            palabras_base_domain = wordninja.split(base_domain)  # Dividir en palabras

            # Extraer el título de la página
            title = soup.title.string if soup.title else ""
            return int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))

        def check_domain_with_copyright():
            """Verificar si el dominio aparece con copyright."""
            # Extraer el dominio base completo (host)
            extracted = tldextract.extract(url)
            domain_full = f"{extracted.domain}.{extracted.suffix}"

            # Buscar símbolos de copyright en el contenido
            match = re.search(r'(©|\(c\)|&copy;|&trade;|&reg;)', response.text)
            if match:
                # Extraer contexto cercano al símbolo
                context_range = 300
                start = max(0, match.start() - context_range)
                end = match.end() + context_range
                _copyright = response.text[start:end].lower()
                if domain_full.lower() in _copyright:
                    return 1
            return 0

        # Ejecutar tareas en paralelo
        with ThreadPoolExecutor() as executor:
            future_hyperlinks = executor.submit(calculate_hyperlinks)
            future_domain_in_title = executor.submit(check_domain_in_title)
            future_domain_with_copyright = executor.submit(check_domain_with_copyright)

            nb_hyperlinks = future_hyperlinks.result()
            domain_in_title = future_domain_in_title.result()
            domain_with_copyright = future_domain_with_copyright.result()

        # Retornar resultados como diccionario
        return {
            "nb_hyperlinks": nb_hyperlinks,
            "domain_in_title": domain_in_title,
            "domain_with_copyright": domain_with_copyright
        }

    except requests.exceptions.RequestException as e:
        print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }
    except Exception as e:
        print(f"Error general procesando la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

# Ejemplo de uso
result = analyze_page_content("https://escuelamartinpescador.cl/")
print(result)


{'nb_hyperlinks': 73, 'domain_in_title': 1, 'domain_with_copyright': 1}


In [67]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import tldextract
import wordninja
from concurrent.futures import ProcessPoolExecutor

def analyze_page_content(url):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        # Realizar la solicitud GET con el User-Agent
        response = requests.get(url, headers=headers, timeout=10)
        
        # Verificar el código de respuesta HTTP
        if response.status_code != 200:
            return {
                "nb_hyperlinks": 0,
                "domain_in_title": 0,
                "domain_with_copyright": 0
            }

        # Parsear el contenido HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        def calculate_hyperlinks():
            """Calcular la cantidad de enlaces y elementos con src."""
            href_count = len(soup.find_all('a', href=True))  # Contar enlaces <a> con href
            src_count = len(soup.find_all(src=True))  # Contar etiquetas con src (ej., <img>, <script>, etc.)
            return href_count + src_count

        def check_domain_in_title():
            """Verificar si el dominio está en el título."""
            # Extraer el dominio base
            extracted = tldextract.extract(url)
            base_domain = extracted.domain  # Nombre base del dominio (sin TLD)
            palabras_base_domain = wordninja.split(base_domain)  # Dividir en palabras

            # Extraer el título de la página
            title = soup.title.string if soup.title else ""
            return int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))

        def check_domain_with_copyright():
            """Verificar si el dominio aparece con copyright."""
            # Extraer el dominio base completo (host)
            extracted = tldextract.extract(url)
            domain_full = f"{extracted.domain}.{extracted.suffix}"

            # Buscar símbolos de copyright en el contenido
            match = re.search(r'(©|\(c\)|&copy;|&trade;|&reg;)', response.text)
            if match:
                # Extraer contexto cercano al símbolo
                context_range = 300
                start = max(0, match.start() - context_range)
                end = match.end() + context_range
                _copyright = response.text[start:end].lower()
                if domain_full.lower() in _copyright:
                    return 1
            return 0

        # Ejecutar tareas en paralelo
        with ProcessPoolExecutor() as executor:
            future_hyperlinks = executor.submit(calculate_hyperlinks)
            future_domain_in_title = executor.submit(check_domain_in_title)
            future_domain_with_copyright = executor.submit(check_domain_with_copyright)

            # Obtener los resultados de las tareas
            nb_hyperlinks = future_hyperlinks.result()
            domain_in_title = future_domain_in_title.result()
            domain_with_copyright = future_domain_with_copyright.result()

        # Retornar resultados como diccionario
        return {
            "nb_hyperlinks": nb_hyperlinks,
            "domain_in_title": domain_in_title,
            "domain_with_copyright": domain_with_copyright
        }

    except requests.exceptions.RequestException as e:
        print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }
    except Exception as e:
        print(f"Error general procesando la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

# Ejemplo de uso
result = analyze_page_content("https://escuelamartinpescador.cl/")
print(result)


Error general procesando la URL https://escuelamartinpescador.cl/: Can't get local object 'analyze_page_content.<locals>.calculate_hyperlinks'
{'nb_hyperlinks': 0, 'domain_in_title': 0, 'domain_with_copyright': 0}


In [78]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import tldextract
import wordninja
from concurrent.futures import ThreadPoolExecutor

def calculate_hyperlinks(html_content):
    """Calcular la cantidad de enlaces y elementos con src."""
    soup = BeautifulSoup(html_content, 'html.parser')
    href_count = len(soup.find_all('a', href=True))  # Contar enlaces <a> con href
    src_count = len(soup.find_all(src=True))  # Contar etiquetas con src (ej., <img>, <script>, etc.)
    return href_count + src_count

def check_domain_in_title(html_content, url):
    """Verificar si el dominio está en el título."""
    extracted = tldextract.extract(url)
    base_domain = extracted.domain  # Nombre base del dominio (sin TLD)
    palabras_base_domain = wordninja.split(base_domain)  # Dividir en palabras
    soup = BeautifulSoup(html_content, 'html.parser')
    title = soup.title.string if soup.title else ""
    return int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))

def check_domain_with_copyright(html_content, url):
    """Verificar si el dominio aparece con copyright."""
    extracted = tldextract.extract(url)
    domain_full = f"{extracted.domain}.{extracted.suffix}"
    match = re.search(r'(©|\(c\)|&copy;|&trade;|&reg;)', html_content)
    if match:
        context_range = 300
        start = max(0, match.start() - context_range)
        end = match.end() + context_range
        _copyright = html_content[start:end].lower()
        if domain_full.lower() in _copyright:
            return 1
    return 0

def analyze_page_content(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return {
                "nb_hyperlinks": 0,
                "domain_in_title": 0,
                "domain_with_copyright": 0
            }

        html_content = response.text

        # Ejecutar tareas en paralelo con hilos
        with ThreadPoolExecutor() as executor:
            future_hyperlinks = executor.submit(calculate_hyperlinks, html_content)
            future_domain_in_title = executor.submit(check_domain_in_title, html_content, url)
            future_domain_with_copyright = executor.submit(check_domain_with_copyright, html_content, url)

            # Obtener los resultados de las tareas
            nb_hyperlinks = future_hyperlinks.result()
            domain_in_title = future_domain_in_title.result()
            domain_with_copyright = future_domain_with_copyright.result()

        return {
            "nb_hyperlinks": nb_hyperlinks,
            "domain_in_title": domain_in_title,
            "domain_with_copyright": domain_with_copyright
        }

    except requests.exceptions.RequestException as e:
        print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }
    except Exception as e:
        print(f"Error general procesando la URL {url}: {e}")
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

# Ejemplo de uso
result = analyze_page_content("https://csirt.gob.cl/alertas/8fph24-00940/")
print(result)


{'nb_hyperlinks': 74, 'domain_in_title': 1, 'domain_with_copyright': 0}


In [96]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import tldextract
import wordninja
from concurrent.futures import ThreadPoolExecutor

def fetch_page_content(url):
    """Descargar el contenido HTML de la página."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            return response.text
        else:
            return None
    except requests.exceptions.RequestException:
        return None


def calculate_hyperlinks(html_content):
    """Calcular la cantidad de enlaces y elementos con src."""
    soup = BeautifulSoup(html_content, 'html.parser')
    href_count = len(soup.find_all('a', href=True))  # Contar enlaces <a> con href
    src_count = len(soup.find_all(src=True))  # Contar etiquetas con src (ej., <img>, <script>, etc.)
    return href_count + src_count


def check_domain_in_title(html_content, url):
    """Verificar si el dominio está en el título."""
    extracted = tldextract.extract(url)
    base_domain = extracted.domain  # Nombre base del dominio (sin TLD)
    palabras_base_domain = wordninja.split(base_domain)  # Dividir en palabras
    soup = BeautifulSoup(html_content, 'html.parser')
    title = soup.title.string if soup.title else ""
    return int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))


def check_domain_with_copyright(html_content, url):
    """Verificar si el dominio aparece con copyright."""
    extracted = tldextract.extract(url)
    domain_full = f"{extracted.domain}.{extracted.suffix}"
    match = re.search(r'(©|\(c\)|&copy;|&trade;|&reg;)', html_content)
    if match:
        context_range = 300
        start = max(0, match.start() - context_range)
        end = match.end() + context_range
        _copyright = html_content[start:end].lower()
        if domain_full.lower() in _copyright:
            return 1
    return 0


def analyze_page_content(url):
    html_content = fetch_page_content(url)

    if not html_content:
        return {
            "nb_hyperlinks": 0,
            "domain_in_title": 0,
            "domain_with_copyright": 0
        }

    # Ejecutar tareas en paralelo con hilos
    with ThreadPoolExecutor() as executor:
        future_hyperlinks = executor.submit(calculate_hyperlinks, html_content)
        future_domain_in_title = executor.submit(check_domain_in_title, html_content, url)
        future_domain_with_copyright = executor.submit(check_domain_with_copyright, html_content, url)

        # Obtener los resultados de las tareas
        nb_hyperlinks = future_hyperlinks.result()
        domain_in_title = future_domain_in_title.result()
        domain_with_copyright = future_domain_with_copyright.result()

    return {
        "nb_hyperlinks": nb_hyperlinks,
        "domain_in_title": domain_in_title,
        "domain_with_copyright": domain_with_copyright
    }


# Ejemplo de uso
result = analyze_page_content("https://escuelamartinpescador.cl/")
print(result)


{'nb_hyperlinks': 73, 'domain_in_title': 1, 'domain_with_copyright': 1}


In [102]:
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup

def check_google_index(url):
    """
    Verifica si un dominio está indexado en Google utilizando una consulta dork.
    """
    # Extraer el dominio
    parsed_url = urlparse(url)
    domain = parsed_url.netloc

    # Construir la consulta dork
    query = f"site:{domain}"
    google_url = f"https://www.google.com/search?q={query}"

    # Configurar headers
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        # Realizar la solicitud GET con un timeout reducido
        response = requests.get(google_url, headers=headers, timeout=5)
        
        # Verificar el código de respuesta HTTP
        if response.status_code == 200:
            # Parsear el HTML con BeautifulSoup
            soup = BeautifulSoup(response.text, 'lxml')

            # Buscar resultados válidos (enlaces)
            search_results = soup.find_all('a', href=True)
            for link in search_results:
                href = link['href']
                # Verificar si el dominio aparece en los resultados
                if domain in href:
                    return 1  # Encontrado en los resultados

            # Si no se encuentra el dominio
            return 0
        else:
            print(f"Error HTTP al consultar Google: {response.status_code}")
            return 0

    except requests.exceptions.RequestException as e:
        print(f"Error en la solicitud: {e}")
        return 0  # Asumimos no indexado en caso de error

# Ejemplo de uso
print(check_google_index("https://escuelamartinpescador.cl"))
print(check_google_index("https://dominioinexistente123456.com"))



FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?

In [ ]:
# Definir todas las funciones auxiliares
def check_google_index(url):
    parsed_url = urlparse(url)
    domain = parsed_url.netloc
    query = f"site:{domain}"
    google_url = f"https://www.google.com/search?q={query}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"}
    try:
        response = requests.get(google_url, headers=headers, timeout=10)
        if response.status_code == 200 and domain in response.text:
            return 1
        return 0
    except Exception:
        return 0


1
1


In [107]:
import concurrent.futures
import time
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import whois
from publicsuffix2 import get_sld
import dns.resolver
import tldextract
import wordninja
import re

# Definir todas las funciones auxiliares
def check_google_index(url):
    parsed_url = urlparse(url)
    domain = parsed_url.netloc
    query = f"site:{domain}"
    google_url = f"https://www.google.com/search?q={query}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"}
    try:
        response = requests.get(google_url, headers=headers, timeout=10)
        if response.status_code == 200 and domain in response.text:
            return 1
        return 0
    except Exception:
        return 0

def analyze_page_content(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return {"nb_hyperlinks": 0, "domain_in_title": 0, "domain_with_copyright": 0}
        soup = BeautifulSoup(response.text, 'html.parser')
        href_count = len(soup.find_all('a', href=True))
        src_count = len(soup.find_all(src=True))
        nb_hyperlinks = href_count + src_count
        extracted = tldextract.extract(url)
        base_domain = extracted.domain
        palabras_base_domain = wordninja.split(base_domain)
        title = soup.title.string if soup.title else ""
        domain_in_title = int(any(palabra.lower() in title.lower() for palabra in palabras_base_domain))
        domain_full = f"{extracted.domain}.{extracted.suffix}"
        match = re.search(r'(©|\(c\)|&copy;|&trade;|&reg;)', response.text)
        domain_with_copyright = 0
        if match:
            context_range = 300
            start = max(0, match.start() - context_range)
            end = match.end() + context_range
            _copyright = response.text[start:end].lower()
            if domain_full.lower() in _copyright:
                domain_with_copyright = 1
        return {"nb_hyperlinks": nb_hyperlinks, "domain_in_title": domain_in_title, "domain_with_copyright": domain_with_copyright}
    except Exception:
        return {"nb_hyperlinks": 0, "domain_in_title": 0, "domain_with_copyright": 0}

def analyze_domain_whois(url):
    try:
        parsed_url = urlparse(url)
        domain = parsed_url.netloc
        if domain.startswith("www."):
            domain = domain[4:]
        domain = get_sld(domain)
        whois_info = whois.whois(domain)
        hostname = whois_info.domain_name
        is_registered = 0
        if isinstance(hostname, list):
            is_registered = int(any(re.search(host.lower(), domain) for host in hostname))
        elif hostname and re.search(hostname.lower(), domain):
            is_registered = 1
        creation_date = whois_info.creation_date
        domain_age = -1
        if creation_date:
            if isinstance(creation_date, list):
                creation_date = creation_date[0]
            if isinstance(creation_date, str):
                creation_date = datetime.strptime(creation_date.split(" ")[0], "%Y-%m-%d")
            if isinstance(creation_date, datetime):
                domain_age = (datetime.now() - creation_date).days
        return {"domain_age": domain_age, "is_registered": is_registered}
    except Exception:
        return {"domain_age": -1, "is_registered": 0}

def count_nb_qm(url): return url.count('?')
def count_nb_www(url): return url.lower().count('www')
def ratio_digits_host(url):
    try:
        parsed_url = urlparse(url)
        host = parsed_url.netloc
        num_digitos = sum(c.isdigit() for c in host)
        longitud_host = len(host)
        return round(num_digitos / longitud_host, 6) if longitud_host > 0 else 0
    except Exception:
        return 0
def count_nb_hyphens(url): return url.count('-')
def count_nb_dots(url): return url.count('.')
def count_nb_eq(url): return url.count('=')
def count_length_hostname(url): return len(urlparse(url).hostname or "")
def ratio_digits_url(url):
    num_digitos = len(re.findall(r'\d', url))
    longitud_url = len(url)
    return round(num_digitos / longitud_url, 6) if longitud_url > 0 else 0
def https_token(url):
    try:
        return 1 if urlparse(url).scheme.lower() == 'https' else 0
    except Exception:
        return 0
def dns_record(url):
    try:
        parsed_url = urlparse(url)
        domain = parsed_url.netloc
        if domain.startswith("www."):
            domain = domain[4:]
        domain = get_sld(domain)
        nameservers = dns.resolver.resolve(domain, 'NS')
        return 1 if len(nameservers) > 0 else 0
    except Exception:
        return 0
def tld_in_subdomain(url):
    try:
        hostname = urlparse(url).hostname or ""
        parts = hostname.split('.')
        if len(parts) > 2:
            tld = parts[-1]
            subdomain = ".".join(parts[:-2])
            return int(tld in subdomain or subdomain == tld)
        return 0
    except Exception:
        return 0
def prefix_suffix(url): return int(bool(re.findall(r"https?://[^\-]+-[^\-]+/", url)))

# Implementación con hilos
def extract_features_url_analyze(url):
    features = {}
    timings = {}
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {
            'google_index': executor.submit(check_google_index, url),
            'content_analysis': executor.submit(analyze_page_content, url),
            'whois_analysis': executor.submit(analyze_domain_whois, url),
            'nb_qm': executor.submit(count_nb_qm, url),
            'nb_www': executor.submit(count_nb_www, url),
            'ratio_digits_host': executor.submit(ratio_digits_host, url),
            'nb_hyphens': executor.submit(count_nb_hyphens, url),
            'nb_dots': executor.submit(count_nb_dots, url),
            'nb_eq': executor.submit(count_nb_eq, url),
            'length_hostname': executor.submit(count_length_hostname, url),
            'ratio_digits_url': executor.submit(ratio_digits_url, url),
            'https_token': executor.submit(https_token, url),
            'dns_record': executor.submit(dns_record, url),
            'tld_in_subdomain': executor.submit(tld_in_subdomain, url),
            'prefix_suffix': executor.submit(prefix_suffix, url)
        }
        results = {key: future.result() for key, future in futures.items()}
    features.update(results)
    content_results = results['content_analysis']
    whois_results = results['whois_analysis']
    features.update({
        'nb_hyperlinks': content_results['nb_hyperlinks'],
        'domain_in_title': content_results['domain_in_title'],
        'domain_with_copyright': content_results['domain_with_copyright'],
        'domain_age': whois_results['domain_age'],
        'whois_registered_domain': whois_results['is_registered']
    })
    return features

print(extract_features_url_analyze(url))

{'google_index': 1, 'content_analysis': {'nb_hyperlinks': 240, 'domain_in_title': 1, 'domain_with_copyright': 0}, 'whois_analysis': {'domain_age': 9913, 'is_registered': 1}, 'nb_qm': 0, 'nb_www': 0, 'ratio_digits_host': 0.0, 'nb_hyphens': 0, 'nb_dots': 1, 'nb_eq': 0, 'length_hostname': 9, 'ratio_digits_url': 0.0, 'https_token': 1, 'dns_record': 1, 'tld_in_subdomain': 0, 'prefix_suffix': 0, 'nb_hyperlinks': 240, 'domain_in_title': 1, 'domain_with_copyright': 0, 'domain_age': 9913, 'whois_registered_domain': 1}
